# PrecisionMiner Workflow

This notebook demonstrates a complete extraction workflow on controlled paper sections. Use it to understand the workflow contract before running over PDFs.


## Setup And Evidence Index

The workflow retrieves evidence from a vector store and sends that evidence to a generator that returns JSON.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

In [ ]:
sample_papers = nb.sample_papers()
list(sample_papers)

In [ ]:
embedder = nb.TinyKeywordEmbedder()
embedder.model_name, embedder.dim

In [ ]:
# Indexing is covered in notebooks 01 and 02; here it is one helper call so the
# notebook can stay focused on the extraction workflow.
vdb, retriever = nb.build_index(WORK_DIR / "index", sample_papers, embedder)

len(vdb.get_points()), vdb.get_embedding_model()

## Configure The Extraction Task

`FindDataSourcesConfig` provides retrieval prompts and the JSON schema used by the response parser.


In [ ]:
from episcope.workflows.precision_miner import FindDataSourcesConfig

strategy_name = "demo-sections"
academic_db = nb.build_academic_db(sample_papers, strategy_name)

config = FindDataSourcesConfig(top_k=3)
config.retrieval_templates[:2], config.top_k

## Provide A Schema-Compatible Generator

Replace this class with `LLMGenerator` when you want model-backed extraction. The
important contract is that `generate` returns JSON matching `ExtractionResult`.

Note the `**kwargs` in the signature — it is not decoration. `config.structured_output`
defaults to `"schema"`, so the workflow calls the generator with
`response_schema=ExtractionResultSchema`, which makes a real `LLMGenerator`
constrain decoding through the provider's native structured-output feature. The
demo generator below accepts and ignores it. Set `structured_output="json"` or
`"off"` on the config if a provider or model does not support it.

In [ ]:
import json
from typing import Any, Callable, Optional, Sequence

from episcope.rag.generation.base import Generator
from episcope.rag.provenance import Evidence, Provenance


class FixedExtractionGenerator(Generator):
    model_id = "fixed-extraction-generator"

    def generate(
        self,
        contexts: Sequence[Any],
        *,
        question: Optional[str] = None,
        message_builder: Optional[Callable[..., Any]] = None,
        **kwargs: Any,
    ) -> Provenance:
        contexts = list(contexts)
        raw_text = contexts[0].text if contexts else ""
        payload = {
            "description": "The paper uses a named registry and public repository evidence.",
            "items": [
                {
                    "name": "National Hospital Registry",
                    "url": None,
                    "explanation": "The registry is named as the source of patient records and outcomes.",
                    "raw_text": raw_text,
                }
            ],
        }
        evidences = [
            Evidence(
                paper_id=getattr(ctx, "paper_id", ""),
                snippet=getattr(ctx, "text", ""),
                section=getattr(ctx, "section_type", None),
                model_id=self.model_id,
                prompt_id="fixed-demo",
            )
            for ctx in contexts
        ]
        return Provenance(answer=json.dumps(payload), evidences=evidences)


## Run And Inspect The Workflow

`run` returns the parsed extraction result. `run_detailed` also exposes prompts, raw generator output, retrieved chunks, and provenance.


In [ ]:
from episcope.workflows import PrecisionMiner

miner = PrecisionMiner(
    retriever=retriever,
    generator=FixedExtractionGenerator(),
    strategy_name=strategy_name,
    config=config,
    academic_db=academic_db,
)

detailed = miner.run_detailed("paper_open_data")
detailed.result.model_dump()


In [ ]:
print("Retrieved chunks")
for chunk in detailed.relevant_chunks:
    print(f"- {chunk.paper_id} | {chunk.section_title} | {chunk.text[:160]}")

print("\nRaw generator response")
print(detailed.trace.raw_llm_response)
